### Restart and Run All Cells

In [1]:
import pandas as pd
from datetime import date, timedelta, datetime
from sqlalchemy import create_engine, text
engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()
year = 2025
quarter = 4
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:03:12 13:38:11


In [3]:
cols = 'name year quarter q_amt_c q_amt_p inc_profit percent'.split()

format_dict = {
                'q_amt':'{:,}','q_amt_c':'{:,}','q_amt_p':'{:,}','inc_profit':'{:,}',
                'yoy_gain':'{:,}','acc_gain':'{:,}',    
                'q_eps':'{:.4f}','y_eps':'{:.4f}','aq_eps':'{:.4f}','ay_eps':'{:.4f}',
                'percent':'{:.2f}%'
              }

In [11]:
sql = f"""
SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = {year}) 
ORDER BY year DESC, quarter DESC"""
print(sql)


SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = 2025) 
ORDER BY year DESC, quarter DESC


In [13]:
dfc = pd.read_sql(sql, conlt)
dfc["Counter"] = 1
dfc_grp = dfc.groupby(["name"], as_index=False).sum()
dfc_grp = dfc_grp[dfc_grp["Counter"] == 4]
dfc_grp.head().style.format(format_dict)

,name,year,quarter,q_amt,Counter
0,3BBIF,8100,10,"10,827,771",4
1,ACE,8100,10,"798,614",4
2,ADVANC,8100,10,"47,885,902",4
4,AH,8100,10,"731,428",4
5,AIE,8100,10,"21,904",4


In [17]:
sql = f"""
SELECT name,year,quarter,q_amt
FROM epss 
WHERE year = {year} - 1  
ORDER BY year DESC, quarter DESC
"""
print(sql)


SELECT name,year,quarter,q_amt
FROM epss 
WHERE year = 2025 - 1  
ORDER BY year DESC, quarter DESC



In [19]:
dfp = pd.read_sql(sql, conlt)
dfp["Counter"] = 1
dfp_grp = dfp.groupby(["name"], as_index=False).sum()
dfp_grp = dfp_grp[dfp_grp["Counter"] == 4]
dfp_grp.head().style.format(format_dict)

,name,year,quarter,q_amt,Counter
0,3BBIF,8096,10,"5,279,119",4
1,ACE,8096,10,"838,717",4
2,ADVANC,8096,10,"35,075,357",4
3,AEONTS,8096,10,"2,860,344",4
4,AH,8096,10,"746,961",4


In [21]:
dfm = pd.merge(dfc_grp, dfp_grp, on="name", suffixes=(["_c", "_p"]), how="inner")
dfm["inc_profit"] = dfm["q_amt_c"] - dfm["q_amt_p"]
dfm["percent"] = round(dfm["inc_profit"] / abs(dfm["q_amt_p"]) * 100, 2)
dfm["year"] = year
dfm["quarter"] = "Q" + str(quarter)
df_percent = dfm[cols]
df_percent.head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","5,279,119","5,548,652",105.11%
1,ACE,2025,Q4,"798,614","838,717","-40,103",-4.78%
2,ADVANC,2025,Q4,"47,885,902","35,075,357","12,810,545",36.52%
3,AH,2025,Q4,"731,428","746,961","-15,533",-2.08%
4,AIE,2025,Q4,"21,904","241,922","-220,018",-90.95%


In [25]:
# Create the SQL query with parameter binding
sqlDel = f"DELETE FROM yr_profits WHERE year = {year} AND quarter = 'Q{quarter}'"
print(sqlDel)
# Execute the query with parameters
rp = conlt.execute(text(sqlDel))

# Print the number of rows affected
print("Rows deleted:", rp.rowcount)

DELETE FROM yr_profits WHERE year = 2025 AND quarter = 'Q4'
Rows deleted: 0


In [27]:
sql = "SELECT name, id FROM tickers"
tickers = pd.read_sql(sql, conlt)
df_ins = pd.merge(df_percent, tickers, on="name", how="inner")
rcds = df_ins.values.tolist()
len(rcds)

201

In [29]:
# Convert DataFrame to list of records
rcds = df_ins.values.tolist()

# Define column names in the same order as values
columns = ['name', 'year', 'quarter', 'latest_amt', 'previous_amt', 'inc_amt', 'inc_pct', 'ticker_id']

# SQL insert statement with named parameters
sql = text("""
    INSERT INTO yr_profits 
    (name, year, quarter, latest_amt, previous_amt, inc_amt, inc_pct, ticker_id)
    VALUES (:name, :year, :quarter, :latest_amt, :previous_amt, :inc_amt, :inc_pct, :ticker_id)
""")

try:
    # Execute inserts
    for rcd in rcds:
        # Convert list to dictionary
        params = dict(zip(columns, rcd))
        conlt.execute(sql, params)
except Exception as e:
    raise e

### End of loop

In [31]:
criteria_1 = df_ins.q_amt_c > 440_000
df_ins.loc[criteria_1, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","5,279,119","5,548,652",105.11%
1,ACE,2025,Q4,"798,614","838,717","-40,103",-4.78%
2,ADVANC,2025,Q4,"47,885,902","35,075,357","12,810,545",36.52%
3,AH,2025,Q4,"731,428","746,961","-15,533",-2.08%
5,AIMIRT,2025,Q4,"684,130","948,696","-264,566",-27.89%


In [33]:
criteria_2 = df_ins.q_amt_p > 400_000
df_ins.loc[criteria_2, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","5,279,119","5,548,652",105.11%
1,ACE,2025,Q4,"798,614","838,717","-40,103",-4.78%
2,ADVANC,2025,Q4,"47,885,902","35,075,357","12,810,545",36.52%
3,AH,2025,Q4,"731,428","746,961","-15,533",-2.08%
5,AIMIRT,2025,Q4,"684,130","948,696","-264,566",-27.89%


In [35]:
criteria_3 = df_ins.percent > 10.00
df_ins.loc[criteria_3, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","5,279,119","5,548,652",105.11%
2,ADVANC,2025,Q4,"47,885,902","35,075,357","12,810,545",36.52%
7,AJ,2025,Q4,"-553,329","-724,541","171,212",23.63%
8,AMATA,2025,Q4,"3,148,658","2,482,899","665,759",26.81%
13,ASK,2025,Q4,"531,545","331,797","199,748",60.20%


In [37]:
final_criteria = criteria_1 & criteria_2 & criteria_3
df_ins.loc[final_criteria, cols].sort_values(by=["percent"], ascending=[False]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
53,DIF,2025,Q4,"13,855,643","656,288","13,199,355",2011.21%
69,GULF,2025,Q4,"101,380,618","18,170,334","83,210,284",457.95%
193,VIBHA,2025,Q4,"1,694,894","698,606","996,288",142.61%
167,TFG,2025,Q4,"7,440,837","3,166,564","4,274,273",134.98%
40,CK,2025,Q4,"3,328,223","1,445,903","1,882,320",130.18%


In [39]:
df_ins.loc[final_criteria, cols].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","5,279,119","5,548,652",105.11%
2,ADVANC,2025,Q4,"47,885,902","35,075,357","12,810,545",36.52%
8,AMATA,2025,Q4,"3,148,658","2,482,899","665,759",26.81%
18,BAM,2025,Q4,"1,812,270","1,601,642","210,628",13.15%
23,BCP,2025,Q4,"2,879,701","2,184,088","695,613",31.85%


In [41]:
df_ins.loc[final_criteria, cols].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q4,"10,827,771","5,279,119","5,548,652",105.11%
2,ADVANC,2025,Q4,"47,885,902","35,075,357","12,810,545",36.52%
8,AMATA,2025,Q4,"3,148,658","2,482,899","665,759",26.81%
18,BAM,2025,Q4,"1,812,270","1,601,642","210,628",13.15%
23,BCP,2025,Q4,"2,879,701","2,184,088","695,613",31.85%


In [43]:
conlt.commit()
conlt.close()

In [45]:
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:03:12 13:49:07
